# 01 — Build the county-month analysis panel

**Study:** Wildfire-associated environmental disturbance and H5N1 detection risk in California (2022–2025).
**Unit of analysis:** county-month (58 counties × 48 months).

This notebook assembles the analysis panel:
1. Outcome — ≥1 confirmed H5N1 wild-bird detection in the county-month (USDA APHIS).
2. Disturbance/exposure — 1-month-lagged FIRMS active fire (count, FRP), MODIS NDVI and LST anomalies, PRISM climate, CDL agricultural share.

Out of the box it runs on the built-in **synthetic panel** (`USE_SYNTHETIC = True` in `src/config.py`) so the full pipeline is reproducible with zero downloads. With real data in `data/raw/` (see `data/README.md`), set `USE_SYNTHETIC = False` and rerun — the downstream notebooks are unchanged.

Key design choice: **all disturbance predictors enter at lag 1 month**, matching the conceptual sequence *disturbance → habitat change → waterfowl redistribution → detection*.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # run from notebooks/

import pandas as pd
from src import config
from src.features import add_seasonality, add_transforms, save_panel

## 1. Load outcome + environmental layers

In [2]:
if config.USE_SYNTHETIC:
    from src.synthetic import make_panel
    panel = make_panel(seed=config.RANDOM_STATE)
    panel["_synthetic"] = True
    print("Synthetic panel:", panel.shape)
else:
    raise SystemExit(
        "Set up real data per data/README.md, then build the panel here "
        "(APHIS outcome + FIRMS/MODIS/PRISM/CDL county-month aggregates)."
    )

Synthetic panel: (2784, 21)


## 2. Feature engineering: seasonality and lagged disturbance

In [3]:
panel = add_seasonality(panel)
panel = add_transforms(panel)
save_panel(panel)
panel.head(3)

,county,fips,lat,lon,month,month_idx,month_num,ag_prop,county_effect,fire_count,...,frp_lag1,ndvi_anom_lag1,lst_anom_lag1,y,_synthetic,sin_month,cos_month,is_winter,fire_lag1_log,frp_lag1_log
0,Alameda,06001,41.958729,-120.525759,2022-01,0,1,0.418399,-0.27299,0,...,NaN,0.000000,0.000000,0,True,0.500000,8.660254e-01,1,0.0,0.0
1,Alameda,06001,41.958729,-120.525759,2022-02,1,2,0.418399,-0.27299,0,...,0.0,-0.704100,0.219383,0,True,0.866025,5.000000e-01,1,0.0,0.0
2,Alameda,06001,41.958729,-120.525759,2022-03,2,3,0.418399,-0.27299,0,...,0.0,-0.549059,-3.816766,0,True,1.000000,6.123234e-17,0,0.0,0.0


## 3. Sanity checks

In [4]:
print("Counties:", panel['county'].nunique(), "| Months:", panel['month'].nunique())
print("Prevalence (county-months with >=1 detection): {:.1%}".format(panel['y'].mean()))
panel[['y','fire_count','ndvi_anom','precip','tmean']].describe().round(2)

Counties: 58 | Months: 48
Prevalence (county-months with >=1 detection): 15.7%


,y,fire_count,ndvi_anom,precip,tmean
count,2784.00,2784.00,2784.00,2784.00,2784.00
mean,0.16,0.36,-0.02,56.70,14.64
std,0.36,0.66,0.66,39.39,6.07
min,0.00,0.00,-1.28,0.00,1.31
25%,0.00,0.00,-0.64,23.81,9.31
50%,0.00,0.00,-0.02,55.43,14.50
75%,0.00,1.00,0.59,86.76,19.90
max,1.00,5.00,1.32,180.79,27.66
